# Retail Shopping Assistant

## Notebook 4: Capture Journey Traces

Record every model call the journeys make, as a trace
[AIPerf](https://github.com/ai-dynamo/aiperf) replays against a NIM in
Notebook 5.

**Why a trace**

- Notebook 5 measures a NIM's prefix-cache hit rate, time to first token, and
  throughput on *this agent's* requests.
- Synthetic prompts share no text: they show a hit rate near zero.
- Running the agent against the NIM is not repeatable: each reply changes the
  next request.
- A trace is the same load every run: every concurrency, cache on or off.

**Two machines**

- **This one (Notebook 4):** the full assistant on hosted endpoints. Writes
  `notebook/traces/`, committed to the repo.
- **The GPU machine (Notebook 5):** only the NIM and AIPerf, plus a checkout
  of the repo. No agent, catalog, memory, Phoenix, or UI.

### What the trace holds

**Files** in `notebook/traces/`:

- `journeys.jsonl.xz`: every journey, every call. Notebook 5's input.
- `journeys/<id>.jsonl.xz`, such as `journeys/J01.jsonl.xz`: one journey
  each, from time 0.
- `manifest.jsonl.xz`: recorded tokens and timing per call, line for line.
- `capture.json`: model, journeys, time.

**One line = one model call**, in AIPerf's `mooncake_trace` format:

```json
{
  "session_id": "trace-bbd2499d-J01_wedding_abroad-0",
  "delay": 3652.5,
  "messages": [
    {"role": "system", "content": [{"type": "text", "text": "You are a retail shopping assistant..."}]},
    {"role": "user", "content": "REQUEST ID: ... I'm going to a wedding in Rome in June..."},
    {"role": "assistant", "content": "", "tool_calls": [{"id": "call-...", "type": "function",
      "function": {"name": "activate_shopper_skills_tool", "arguments": "{...}"}}]},
    {"role": "tool", "tool_call_id": "call-...", "content": "SHOPPER_SKILL_ACTIVATION_COMPLETE: ..."}
  ],
  "tools": [{"type": "function", "function": {"name": "search_catalog_tool", "parameters": {...}}}],
  "output_length": 20,
  "extra": {"temperature": 0.0, "parallel_tool_calls": false,
            "chat_template_kwargs": {"enable_thinking": false}}
}
```

- `session_id`: the conversation. AIPerf sends its calls in order.
- `timestamp` (first call) / `delay` (later calls): when to send, in ms. `delay`
  is the agent's own tool time, not the model's.
- `messages`: exactly what the agent sent.
- `tools`: the tools offered. None on the response editor.
- `output_length`: reply length, sent as `max_tokens`.
- `extra`: the rest of the request, including `tool_choice` when forced.

**Left out on purpose:** the model name (Notebook 5 sets it), hosted model
time, reply text (replayed by length), and refused calls.

### In this notebook

1. Confirm the stack
2. Run the journeys
3. Build the trace
4. Check it is complete
5. Read the trace
6. Prefix reuse
7. Save
8. Validate with AIPerf

### Before you start

- **Notebooks 1 to 3 done**, with the deployment still running.
- **Time:** about 35 minutes, mostly running the journeys.

<div class="alert alert-block alert-success">
<b>Tip:</b> When you can, capture on the model you will test.
</div>

## Step 1: Confirm the Stack

In [ ]:
from helpers import CHAIN_SERVER, PHOENIX, status

print(f"Agent:   {CHAIN_SERVER}   status: {status(CHAIN_SERVER + '/health')}")
print(f"Phoenix: {PHOENIX}   status: {status(PHOENIX)}")
!echo "OTEL_EXPORTER_OTLP_ENDPOINT=$(docker exec chain-server printenv OTEL_EXPORTER_OTLP_ENDPOINT)"
!echo "LLM_MODEL=$(docker exec chain-server printenv LLM_MODEL)"

**You should see:**

<pre style="font-size: 0.8em; line-height: 1.35;">
Agent:   http://localhost:8009   status: 200
Phoenix: http://localhost:6006   status: 200
OTEL_EXPORTER_OTLP_ENDPOINT=http://otel-collector:4318
LLM_MODEL=nvidia/nvidia/nemotron-3.5-super-text-preview
</pre>

- Agent not `200`: deploy (Notebook 1, Step 5).
- Phoenix not `200`: `docker compose up -d phoenix otel-collector`.
- Endpoint empty: tracing is off. Unset `OTEL_SDK_DISABLED`, then
  `docker compose up -d chain-server`.

## Step 2: Run the Journeys

- Replays each journey's shopper messages (the Notebook 3 runner). Every model
  call lands in Phoenix.
- `CONCURRENCY = 1`: the hosted endpoint returns 429s at higher concurrency.

<div class="alert alert-block alert-success">
<b>Tip:</b> On your own NIM, raise <code>CONCURRENCY</code>; the trace is the same. For
a quick try, set <code>JOURNEYS = ["J01", "J21"]</code>.
</div>

In [ ]:
import sys

from helpers import new_conversation
from trace_capture import journey_ids, now

JOURNEYS = journey_ids()
CONCURRENCY = 1

label = new_conversation("trace")
since = now()
print(f"Capture {label}: {len(JOURNEYS)} journeys, {CONCURRENCY} at once, from {since}")
!cd .. && {sys.executable} -m tests.evaluation.src.replay --only {",".join(JOURNEYS)} --label {label} --concurrency {CONCURRENCY}

**You should see** one line per journey, then totals:

<pre style="font-size: 0.8em; line-height: 1.35;">
Capture trace-bbd2499d: 25 journeys, 1 at once, from 2026-09-28T14:15:57.690842+00:00
  assistant healthy at http://localhost:8009
  25 runs, one at a time, label trace-bbd2499d
  ok   J01_wedding_abroad
  ok   J02_video_look_full
  ...
  ok   J25_a_size_before_a_product

  pass 20  fail 5  error 0
</pre>

<div class="alert alert-block alert-info">
<b>Note:</b> A few failed checks are fine: the trace records the load the journeys asked
for, not whether they passed.
</div>

**If most fail** with `products_min: 0` from turn 1: the endpoint is refusing
requests (429). Wait, lower `CONCURRENCY`, and rerun.

## Step 3: Build the Trace

- Reads each `ChatOpenAI` span from Phoenix and rebuilds the request it sent.
- Waits for the last spans to arrive. Takes a minute or two.

In [ ]:
from helpers import REPO
from trace_capture import capture

RESULTS = REPO / "tests" / "evaluation" / "results" / "val" / label

lines, manifest, checks, refused = capture(label, since, RESULTS)
print(f"{len(lines)} model calls from {len(checks)} conversations, {refused} refused by the endpoint")

**You should see:**

<pre style="font-size: 0.8em; line-height: 1.35;">
897 model calls from 25 conversations, 0 refused by the endpoint
</pre>

- Refused calls are left out. Any refusal: capture again at lower
  `CONCURRENCY`.

## Step 4: Check It Is Complete

- Compares with the replay runner's own count, turn by turn: calls and prompt
  tokens.

In [ ]:
for check in checks:
    marks = "calls ok " if check["calls_match"] else "CALLS MISSING"
    marks += "  tokens ok" if check["prompt_tokens_match"] else "  TOKENS DIFFER"
    print(f"{check['scenario']:<34} {check['turns']:>3} turns {check['calls']:>4} calls   {marks}   ({check['outcome']})")

complete = not refused and all(check["calls_match"] and check["prompt_tokens_match"] for check in checks)
print()
print("Complete: every call, in its turn, with its prompt" if complete else "Incomplete: see the table below")
if refused:
    print(f"{refused} calls were refused by the endpoint: capture again with a lower CONCURRENCY")

**You should see** `calls ok` and `tokens ok` on every journey:

<pre style="font-size: 0.8em; line-height: 1.35;">
J01_wedding_abroad                  19 turns   85 calls   calls ok   tokens ok   (pass)
J02_video_look_full                 13 turns   59 calls   calls ok   tokens ok   (pass)
...
J25_a_size_before_a_product          2 turns    8 calls   calls ok   tokens ok   (pass)

Complete: every call, in its turn, with its prompt
</pre>

- `CALLS MISSING` on the last journeys: spans still arriving. Rerun Step 3.
- `CALLS MISSING` that stays, or `TOKENS DIFFER`: capture that journey again.

## Step 5: Read the Trace

Three kinds of model call, each turn:

- **Skill selection:** picks the turn's skills. One forced tool.
- **Agent:** tool calls, then the reply. Two or more per turn.
- **Response editor:** checks the reply against tool results. No tools.

In [ ]:
from collections import defaultdict
from statistics import median, quantiles

from trace_capture import kind

prompts = defaultdict(list)
for line, row in zip(lines, manifest, strict=True):
    prompts[kind(line)].append(row["prompt_tokens"])

print(f"{'call':<18}{'calls':>7}{'median prompt':>15}{'p95':>9}{'longest':>9}   tokens")
for name in ("skill selection", "agent", "response editor"):
    tokens = prompts[name]
    p95 = quantiles(tokens, n=20, method="inclusive")[-1]
    print(f"{name:<18}{len(tokens):>7}{median(tokens):>15,.0f}{p95:>9,.0f}{max(tokens):>9,}")

turns = len({(row["session_id"], row["turn"]) for row in manifest})
print()
print(f"{len(lines)} calls over {turns} turns: {len(lines) / turns:.1f} per turn")
print(f"{sum(row['prompt_tokens'] for row in manifest):,} prompt tokens, "
      f"{sum(row['completion_tokens'] for row in manifest):,} completion tokens")

**You should see** (varies a little per run):

<pre style="font-size: 0.8em; line-height: 1.35;">
call                calls  median prompt      p95  longest   tokens
skill selection       210          3,512    6,823    8,233
agent                 475         16,535   21,857   26,668
response editor       212          5,180    9,246   21,278

897 calls over 208 turns: 4.3 per turn
9,907,735 prompt tokens, 145,851 completion tokens
</pre>

One line of the trace, text cut short:

In [ ]:
import json

example = next(line for line in lines if kind(line) == "agent" and len(line["messages"]) >= 6)

def cut(value, size=70):
    text = value if isinstance(value, str) else json.dumps(value)
    return text if len(text) <= size else text[:size] + "..."

print("session_id:   ", example["session_id"])
print("delay (ms):   ", example.get("delay"), "  the agent's time since the previous reply")
print("output_length:", example["output_length"])
print("extra:        ", example["extra"])
print("tools:        ", [tool["function"]["name"] for tool in example["tools"]])
print("messages:")
for message in example["messages"]:
    calls = [call["function"]["name"] for call in message.get("tool_calls", [])]
    print(f"  {message['role']:<10}", cut(message["content"]), f"-> calls {calls}" if calls else "")

**You should see:**

<pre style="font-size: 0.8em; line-height: 1.35;">
session_id:    trace-bbd2499d-J01_wedding_abroad-0
delay (ms):    3652.5   the agent's time since the previous reply
output_length: 20
extra:         {'temperature': 0.0, 'parallel_tool_calls': False, 'chat_template_kwargs': {'enable_thinking': False}}
tools:         ['search_catalog_tool', 'get_product_details_tool', 'resolve_conversation_products_tool', ...]
messages:
  system     [{"type": "text", "text": "You are a retail shopping assistant for the...
  user       REQUEST ID: d1f509d1-19a2-4d7c-a791-03ca15970bb5 ...
  assistant   -> calls ['activate_shopper_skills_tool']
  tool       SHOPPER_SKILL_ACTIVATION_COMPLETE: /shopper/outfit-styling/SKILL.md
  assistant   -> calls ['search_catalog_tool']
  tool       SCOPE 1 (dresses): ...
</pre>

## Step 6: Prefix Reuse

- For each call: the longest start it shares with any earlier call.
- **Within the conversation:** history and the previous turn's prompts.
- **Across conversations:** shared system prompts and tools.

In [ ]:
from trace_capture import prefix_reuse

reuse = prefix_reuse(lines)

def weighted(rows, field):
    return sum(row[field] * row["chars"] for row in rows) / sum(row["chars"] for row in rows)

by_kind = defaultdict(list)
for line, row in zip(lines, reuse, strict=True):
    by_kind[kind(line)].append(row)

print(f"{'call':<18}{'within the conversation':>25}{'across conversations':>23}")
for name in ("skill selection", "agent", "response editor", "all calls"):
    rows = reuse if name == "all calls" else by_kind[name]
    print(f"{name:<18}{weighted(rows, 'from_session'):>25.0%}{weighted(rows, 'from_any'):>23.0%}")

**You should see:**

<pre style="font-size: 0.8em; line-height: 1.35;">
call                within the conversation   across conversations
skill selection                         40%                    46%
agent                                   77%                    86%
response editor                         42%                    49%
all calls                               70%                    79%
</pre>

- Weighted by request length.
- A text estimate and a ceiling: the chat template and cache eviction lower
  it. Notebook 5 measures the real rate against it.

## Step 7: Save

- Writes `notebook/traces/`, replacing the previous capture.
- xz shrinks the trace enough to commit. Python reads it without extra
  libraries.
- No real shopper data or keys: scripted journeys, demo catalog.

<div class="alert alert-block alert-success">
<b>Tip:</b> Commit the trace for Notebook 5:
<code>git add notebook/traces &amp;&amp; git commit -m "Capture journey traces"</code>
</div>

In [ ]:
from trace_capture import TRACES, by_journey, write_jsonl

write_jsonl(TRACES / "journeys.jsonl.xz", lines)
write_jsonl(TRACES / "manifest.jsonl.xz", manifest)
per_journey = by_journey(lines)
for journey, calls in per_journey.items():
    write_jsonl(TRACES / "journeys" / f"{journey}.jsonl.xz", calls)
(TRACES / "capture.json").write_text(json.dumps({
    "label": label,
    "since": since,
    "model": manifest[0]["model"],
    "journeys": JOURNEYS,
    "concurrency": CONCURRENCY,
    "calls": len(lines),
    "complete": complete,
    "format": "aiperf mooncake_trace",
}, indent=2) + "\n")

for path in sorted(TRACES.glob("*.*")):
    print(f"{path.relative_to(REPO)}   {path.stat().st_size / 1e6:.2f} MB")
print()
print(f"{'journey':<9}{'turns':>6}{'calls':>7}{'prompt tokens':>15}")
for journey, calls in per_journey.items():
    rows = [row for row in manifest if f"-{journey}_" in row["session_id"]]
    turns = len({row["turn"] for row in rows})
    print(f"{journey:<9}{turns:>6}{len(calls):>7}{sum(row['prompt_tokens'] for row in rows):>15,}")

**You should see:**

<pre style="font-size: 0.8em; line-height: 1.35;">
notebook/traces/capture.json   0.00 MB
notebook/traces/journeys.jsonl.xz   0.19 MB
notebook/traces/manifest.jsonl.xz   0.02 MB

journey   turns  calls  prompt tokens
J01          19     85      1,243,211
J02          13     59        805,797
...
J25           2      8         70,149
</pre>

## Step 8: Validate with AIPerf

- Runs AIPerf's own schema check here, not first on the GPU machine.
- Install once, the same way as Notebook 5:

```bash
command -v uv || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv venv ~/aiperf --python 3.12
uv pip install --python ~/aiperf/bin/python aiperf==0.13.0
```

<div class="alert alert-block alert-info">
<b>Note:</b> <code>uv</code> needs no <code>sudo</code>. Stock Ubuntu images lack
<code>python3-venv</code>, so <code>python3 -m venv</code> fails there.
</div>

In [ ]:
import subprocess
from pathlib import Path

AIPERF_PYTHON = Path.home() / "aiperf" / "bin" / "python"
VALIDATE = """
import lzma, sys
from aiperf.dataset.loader.models import MooncakeTrace
lines = lzma.open(sys.argv[1], "rt", encoding="utf-8").read().splitlines()
for line in lines:
    MooncakeTrace.model_validate_json(line)
print(f"AIPerf accepts all {len(lines)} lines")
"""

_ = subprocess.run([AIPERF_PYTHON, "-c", VALIDATE, TRACES / "journeys.jsonl.xz"], check=True)

**You should see:**

<pre style="font-size: 0.8em; line-height: 1.35;">
AIPerf accepts all 897 lines
</pre>

- `FileNotFoundError`: install AIPerf.
- `ValidationError`: report the line and field.

## Next: Notebook 5, on the GPU Machine

- Needs: a repo checkout, the NIM, AIPerf.
- Replays `journeys.jsonl.xz` at rising concurrency, cache on and off, and
  reads the NIM's cache counters.

```bash
aiperf profile \
  --model <the NIM's model> --url http://localhost:8000 \
  --endpoint-type chat --streaming \
  --input-file journeys.jsonl --custom-dataset-type mooncake_trace \
  --no-fixed-schedule --concurrency 16 \
  --extra-inputs ignore_eos:true
```

- The model the trace was captured on is in `capture.json`. Requests and
  reply lengths replay exactly on any model.
- To check a different model would steer the same way: point this machine's
  agent at the NIM (`LLM_BASE_URL`) and capture a few journeys.